In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
FULL PIPELINE: Rekalibrasi Ruang Laten MCU-Quake untuk Indonesia
Gabungan dari semua script: ekstraksi, rekalibrasi, validasi, & deployment.
"""

import os
import sys
import json
import pickle
import logging
import numpy as np
from scipy.stats import gaussian_kde, norm
from tensorflow import keras
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. KONFIGURASI
# =============================================

# Path ke data dan model
JSON_PATH = "/Volumes/Extreme SSD/unduhan_waveform_merged/extracted_data.json"
MODEL_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20"

# Path output
OUTPUT_DIR = "/Volumes/Extreme SSD/mcu_quake_recalibration"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Path log
LOG_FILE = os.path.join(OUTPUT_DIR, "recalibration.log")

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger(__name__)

# Tambahkan path Library
sys.path.append('/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying')
from Library import utils

# =============================================
# 2. FUNGSI UTAMA
# =============================================

def load_model_and_data():
    """Load model MCU-Quake dan data Indonesia."""
    logger.info("Loading model MCU-Quake...")
    model = keras.models.load_model(MODEL_PATH)
    logger.info("✅ Model loaded.")
    
    logger.info("Loading data Indonesia...")
    with open(JSON_PATH, 'r') as f:
        data = json.load(f)
    logger.info(f"✅ Total event: {len(data)}")
    return model, data

def extract_embeddings(model, data):
    """Ekstrak embedding dari data Indonesia."""
    logger.info("Extracting embeddings...")
    
    embeddings_no = []
    embeddings_le = []
    
    for key, record in tqdm(data.items(), desc="Ekstraksi"):
        label = record.get('type', 'unknown')
        z_signal = np.array(record.get('Z', []))
        if len(z_signal) != 700:
            continue
        try:
            emb = utils.latent_codes_1D(z_signal, model)
            if label in ['noise', 'NO']:
                embeddings_no.append(emb)
            elif label in ['se', 'le', 'LE']:
                embeddings_le.append(emb)
        except Exception as e:
            logger.debug(f"Error on {key}: {e}")
            continue
    
    emb_no = np.array(embeddings_no).flatten()
    emb_le = np.array(embeddings_le).flatten()
    
    logger.info(f"✅ Noise embeddings: {len(emb_no)} samples")
    logger.info(f"✅ Earthquake embeddings: {len(emb_le)} samples")
    return emb_no, emb_le

def recalibrate_kde(emb_no, emb_le):
    """Rekalibrasi KDE dan hitung statistik."""
    logger.info("Recalibrating KDE...")
    
    mean_no = np.mean(emb_no) if len(emb_no) > 0 else 0
    std_no = np.std(emb_no) if len(emb_no) > 0 else 1
    mean_le = np.mean(emb_le) if len(emb_le) > 0 else 0
    std_le = np.std(emb_le) if len(emb_le) > 0 else 1
    
    logger.info(f"  Noise: mean={mean_no:.4f}, std={std_no:.4f}")
    logger.info(f"  Earthquake: mean={mean_le:.4f}, std={std_le:.4f}")
    
    kde_no = gaussian_kde(emb_no) if len(emb_no) > 0 else None
    kde_le = gaussian_kde(emb_le) if len(emb_le) > 0 else None
    
    return mean_no, std_no, mean_le, std_le, kde_no, kde_le

def save_parameters(mean_no, std_no, mean_le, std_le, kde_no, kde_le, output_dir):
    """Simpan parameter KDE ke JSON, pickle, dan C++ header."""
    logger.info("Saving parameters...")
    
    # JSON
    params = {
        "NO": {"mean": float(mean_no), "std": float(std_no), "n_samples": len(emb_no)},
        "LE": {"mean": float(mean_le), "std": float(std_le), "n_samples": len(emb_le)},
        "reference": {"NO": {"mean": -5.01, "std": 1.14}, "LE": {"mean": 1.01, "std": 0.49}}
    }
    json_path = os.path.join(output_dir, "kde_indonesia_params.json")
    with open(json_path, 'w') as f:
        json.dump(params, f, indent=2)
    logger.info(f"  ✅ JSON saved: {json_path}")
    
    # Pickle
    pkl_path = os.path.join(output_dir, "kde_indonesia.pkl")
    with open(pkl_path, 'wb') as f:
        pickle.dump({"kde_no": kde_no, "kde_le": kde_le}, f)
    logger.info(f"  ✅ Pickle saved: {pkl_path}")
    
    # C++ Header
    h_path = os.path.join(output_dir, "kde_indonesia.h")
    h_content = generate_cpp_header(mean_no, std_no, mean_le, std_le)
    with open(h_path, 'w') as f:
        f.write(h_content)
    logger.info(f"  ✅ C++ header saved: {h_path}")

def generate_cpp_header(mean_no, std_no, mean_le, std_le):
    """Generate C++ header untuk deployment."""
    return f"""#ifndef MCU_QUAKE_KDE_INDONESIA_H
#define MCU_QUAKE_KDE_INDONESIA_H

#include <Arduino.h>
#include <math.h>

// =============================================
// PARAMETER KDE HASIL REKALIBRASI INDONESIA
// =============================================

const float KDE_NO_MEAN = {mean_no:.6f}f;
const float KDE_NO_STD = {std_no:.6f}f;
const float KDE_LE_MEAN = {mean_le:.6f}f;
const float KDE_LE_STD = {std_le:.6f}f;

const float UUSS_NO_MEAN = -5.01f;
const float UUSS_NO_STD = 1.14f;
const float UUSS_LE_MEAN = 1.01f;
const float UUSS_LE_STD = 0.49f;

// =============================================
// FUNGSI LIKELIHOOD
// =============================================

float gaussian_likelihood(float x, float mean, float std) {{
    float exponent = -0.5f * powf((x - mean) / std, 2);
    return (1.0f / (std * sqrtf(2.0f * M_PI))) * expf(exponent);
}}

// =============================================
// INFERENSI DENGAN KDE INDONESIA
// =============================================

int predict_with_kde_indonesia(float embedding) {{
    float p_noise = gaussian_likelihood(embedding, KDE_NO_MEAN, KDE_NO_STD);
    float p_earthquake = gaussian_likelihood(embedding, KDE_LE_MEAN, KDE_LE_STD);
    return (p_earthquake > p_noise) ? 1 : 0;
}}

#endif // MCU_QUAKE_KDE_INDONESIA_H
"""

def validate_performance(model, data, kde_no, kde_le):
    """Uji performa KDE baru pada data Indonesia."""
    logger.info("Validating performance...")
    
    true_labels = []
    pred_labels = []
    
    for key, record in tqdm(data.items(), desc="Validasi"):
        label = record.get('type', 'unknown')
        z_signal = np.array(record.get('Z', []))
        if len(z_signal) != 700:
            continue
        if label not in ['noise', 'NO', 'se', 'le', 'LE']:
            continue
        try:
            emb = utils.latent_codes_1D(z_signal, model)
            if kde_no is not None and kde_le is not None:
                p_no = kde_no(emb)
                p_le = kde_le(emb)
                pred = 'le' if p_le > p_no else 'noise'
            else:
                pred = 'le' if emb > (mean_no + mean_le) / 2 else 'noise'
            true = 'le' if label in ['se', 'le', 'LE'] else 'noise'
            true_labels.append(true)
            pred_labels.append(pred)
        except Exception as e:
            continue
    
    correct = sum([1 for t, p in zip(true_labels, pred_labels) if t == p])
    accuracy = correct / len(true_labels) if len(true_labels) > 0 else 0
    logger.info(f"✅ Accuracy: {accuracy*100:.2f}% ({correct}/{len(true_labels)})")
    return accuracy, true_labels, pred_labels

def plot_results(emb_no, emb_le, kde_no, kde_le, mean_no, std_no, mean_le, std_le, accuracy, output_dir):
    """Buat visualisasi perbandingan."""
    logger.info("Generating visualizations...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Histogram
    ax1 = axes[0, 0]
    x_range = np.linspace(-10, 6, 500)
    if len(emb_no) > 0:
        ax1.hist(emb_no, bins=50, alpha=0.5, label=f'Noise (n={len(emb_no)})', color='gray')
    if len(emb_le) > 0:
        ax1.hist(emb_le, bins=50, alpha=0.5, label=f'EQ (n={len(emb_le)})', color='orange')
    ax1.set_xlabel('Embedding Value')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Distribusi Embedding Indonesia')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. KDE overlay
    ax2 = axes[0, 1]
    if kde_no is not None:
        ax2.plot(x_range, kde_no(x_range), label='NO (Indonesia)', color='gray', linewidth=2)
    if kde_le is not None:
        ax2.plot(x_range, kde_le(x_range), label='LE (Indonesia)', color='orange', linewidth=2)
    ref_no = norm.pdf(x_range, -5.01, 1.14)
    ref_le = norm.pdf(x_range, 1.01, 0.49)
    ax2.plot(x_range, ref_no, 'k--', label='NO (UUSS ref)', linewidth=1.5, alpha=0.6)
    ax2.plot(x_range, ref_le, 'r--', label='LE (UUSS ref)', linewidth=1.5, alpha=0.6)
    ax2.axvline(-5.01, color='black', linestyle=':', alpha=0.5)
    ax2.axvline(1.01, color='black', linestyle=':', alpha=0.5)
    ax2.set_xlabel('Embedding Value')
    ax2.set_ylabel('Density')
    ax2.set_title('KDE Indonesia vs Referensi UUSS')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Boxplot
    ax3 = axes[1, 0]
    box_data = []
    box_labels = []
    if len(emb_no) > 0:
        box_data.append(emb_no)
        box_labels.append(f'Noise (n={len(emb_no)})')
    if len(emb_le) > 0:
        box_data.append(emb_le)
        box_labels.append(f'EQ (n={len(emb_le)})')
    if box_data:
        ax3.boxplot(box_data, labels=box_labels, patch_artist=True)
        ax3.axhline(-5.01, color='gray', linestyle='--', label='UUSS Noise mean')
        ax3.axhline(1.01, color='orange', linestyle='--', label='UUSS EQ mean')
    ax3.set_ylabel('Embedding Value')
    ax3.set_title('Boxplot Embedding Indonesia')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 4. Summary table
    ax4 = axes[1, 1]
    ax4.axis('tight')
    ax4.axis('off')
    summary_data = [
        ['Parameter', 'Noise', 'Earthquake'],
        ['Mean (Indonesia)', f'{mean_no:.3f}', f'{mean_le:.3f}'],
        ['Std (Indonesia)', f'{std_no:.3f}', f'{std_le:.3f}'],
        ['N samples', f'{len(emb_no)}', f'{len(emb_le)}'],
        ['Mean (UUSS ref)', '-5.010', '1.010'],
        ['Std (UUSS ref)', '1.140', '0.490'],
        ['Accuracy KDE baru', f'{accuracy*100:.2f}%', ''],
    ]
    table = ax4.table(cellText=summary_data, cellLoc='center', loc='center',
                      colColours=['#4472C4', '#2ecc71', '#e74c3c'])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.6)
    ax4.set_title('Ringkasan Rekalibrasi', fontsize=14, pad=20)
    
    plt.tight_layout()
    plot_path = os.path.join(output_dir, 'recalibration_analysis.png')
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    logger.info(f"  ✅ Plot saved: {plot_path}")

# =============================================
# 3. MAIN
# =============================================

if __name__ == "__main__":
    try:
        # 1. Load
        model, data = load_model_and_data()
        
        # 2. Extract embeddings
        emb_no, emb_le = extract_embeddings(model, data)
        
        # 3. Recalibrate KDE
        mean_no, std_no, mean_le, std_le, kde_no, kde_le = recalibrate_kde(emb_no, emb_le)
        
        # 4. Save parameters
        save_parameters(mean_no, std_no, mean_le, std_le, kde_no, kde_le, OUTPUT_DIR)
        
        # 5. Validate
        accuracy, _, _ = validate_performance(model, data, kde_no, kde_le)
        
        # 6. Plot
        plot_results(emb_no, emb_le, kde_no, kde_le, mean_no, std_no, mean_le, std_le, accuracy, OUTPUT_DIR)
        
        logger.info("\n" + "="*70)
        logger.info("✅ REKALIBRASI SELESAI!")
        logger.info(f"📁 Output folder: {OUTPUT_DIR}")
        logger.info("="*70)
        
    except Exception as e:
        logger.error(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        sys.exit(1)